### Off-Policy Hyperparametertuning of the Policies

In [1]:
import pandas as pd, numpy as np, plotly.express as px, plotly.graph_objects as go

#### Load Data

In [2]:
# Load policy train data
policy_train_0 = pd.read_csv("./Data/policy_train.csv")
policy_train_005 = pd.read_csv("./Data/policy_train_all_feat_noise005.csv")
policy_train_01 = pd.read_csv("./Data/policy_train_all_feat_noise01.csv")
policy_train_015 = pd.read_csv("./Data/policy_train_all_feat_noise015.csv")
policy_train_02 = pd.read_csv("./Data/policy_train_all_feat_noise02.csv")
policy_train_025 = pd.read_csv("./Data/policy_train_all_feat_noise025.csv")
policy_train_03 = pd.read_csv("./Data/policy_train_all_feat_noise03.csv")
policy_train_035 = pd.read_csv("./Data/policy_train_all_feat_noise035.csv")

policy_test_0 = pd.read_csv("./Data/policy_test.csv")
policy_test_005 = pd.read_csv("./Data/policy_test_all_feat_noise005.csv")
policy_test_01 = pd.read_csv("./Data/policy_test_all_feat_noise01.csv")
policy_test_015 = pd.read_csv("./Data/policy_test_all_feat_noise015.csv")
policy_test_02 = pd.read_csv("./Data/policy_test_all_feat_noise02.csv")
policy_test_025 = pd.read_csv("./Data/policy_test_all_feat_noise025.csv")
policy_test_03 = pd.read_csv("./Data/policy_test_all_feat_noise03.csv")
policy_test_035 = pd.read_csv("./Data/policy_test_all_feat_noise035.csv")

In [3]:
def reshape_dataframe(df):
    array = df.to_numpy()  
    num_cols = array.shape[1]
    new_length = (array.shape[0] // 24) * 24  
    array = array[:new_length, :]  
    reshaped_array = array.reshape(new_length // 24, 24, num_cols).transpose(2, 0, 1)

    return reshaped_array

In [4]:
reshaped_policy_train_0 = reshape_dataframe(policy_train_0)
reshaped_policy_train_005 = reshape_dataframe(policy_train_005)
reshaped_policy_train_01 = reshape_dataframe(policy_train_01)
reshaped_policy_train_015 = reshape_dataframe(policy_train_015)
reshaped_policy_train_02 = reshape_dataframe(policy_train_02)
reshaped_policy_train_025 = reshape_dataframe(policy_train_025)
reshaped_policy_train_03 = reshape_dataframe(policy_train_03)
reshaped_policy_train_035 = reshape_dataframe(policy_train_035)

reshaped_policy_test_0 = reshape_dataframe(policy_test_0)
reshaped_policy_test_005 = reshape_dataframe(policy_test_005)
reshaped_policy_test_01 = reshape_dataframe(policy_test_01)
reshaped_policy_test_015 = reshape_dataframe(policy_test_015)
reshaped_policy_test_02 = reshape_dataframe(policy_test_02)
reshaped_policy_test_025 = reshape_dataframe(policy_test_025)
reshaped_policy_test_03 = reshape_dataframe(policy_test_03)
reshaped_policy_test_035 = reshape_dataframe(policy_test_035)

### Optimizing the BLSH policy

#### Off-policy on the historical training data

In [5]:
from Models.EnergyStorageModel import EnergyStorageModel as ESM
from Models.BaseClasses.Util import grid_search_PFA

initial_state = {"energy_amount": 300, "price": reshaped_policy_train_0[0][0]}
init_args = {"eta": 0.95, "Rmax": 600, "max_load_per_hour": 50}
model_name = "cnf-24"
exog_params = {"hist_price": reshaped_policy_train_0[0][1:]}
T = len(reshaped_policy_train_0[0][1:])
t0 = 0

BLSH_train_model = ESM(
    model_name="cnf-24",
    S0=initial_state,
    init_args=init_args,
    exog_params=exog_params,
    T=T,
    t0=t0,
    seed=0
)

In [18]:
from Models.Policies.PFA import BuyLowSellHigh as BLSH

# fill values for the hyperparameters we overwrite them in the grid search
theta_low = 10
theta_high = 20

BLSH_train_policy = BLSH(
    policy_name="BLSH-Hyperparam-Search",
    model=BLSH_train_model,
    theta_low=theta_low,
    theta_high=theta_high,
    verbose=False
)

In [19]:
prices = reshaped_policy_train_0[0].flatten()

fig = px.histogram(prices, nbins=100, title='Distribution of Prices')
fig.update_layout(xaxis_title='Prices', yaxis_title='Frequency')
fig.show()

In [20]:
theta_low_values = np.linspace(300, 40, 53)  
theta_high_values = np.linspace(60, 300, 49)  

valid_combinations = [(low, high) for low in theta_low_values for high in theta_high_values if low < high]

BLSH_grid = {
    "theta_low": np.array([combo[0] for combo in valid_combinations]),
    "theta_high": np.array([combo[1] for combo in valid_combinations])
}

result = grid_search_PFA(BLSH_grid, BLSH_train_policy, n_iterations=1)
print("Best Parameters:", result["best_parameters"])
print("Best Performance:", result["best_performance"])

Best Parameters: {'theta_low': np.float64(295.0), 'theta_high': np.float64(300.0)}
Best Performance: 4674.991915789476


In [21]:
initial_state = {"energy_amount": 300, "price": reshaped_policy_test_0[0][0]}
init_args = {"eta": 0.95, "Rmax": 600, "max_load_per_hour": 50}
model_name = "cnf-24"
exog_params = {"hist_price": reshaped_policy_test_0[0][1:]}
T = len(reshaped_policy_test_0[0][1:])
t0 = 0

BLSH_test_model = ESM(
    model_name="cnf-24",
    S0=initial_state,
    init_args=init_args,
    exog_params=exog_params,
    T=T,
    t0=t0,
    seed=0
)

theta_low = 295
theta_high = 300

BLSH_test_policy = BLSH(
    policy_name="BLSH-Hyperparam-Search",
    model=BLSH_test_model,
    theta_low=theta_low,
    theta_high=theta_high,
    verbose=False
)

BLSH_test_policy.run_policy()

302.94843026315783

#### On-policy one scneario

In [22]:
initial_state = {"energy_amount": 300, "price": reshaped_policy_test_0[0][0]}
init_args = {"eta": 0.95, "Rmax": 600, "max_load_per_hour": 50}
model_name = "cnf-24"
exog_params = {"hist_price": reshaped_policy_test_0[0][1:]}
T = len(reshaped_policy_test_0[0][1:])
t0 = 0

BLSH_test_model = ESM(
    model_name="cnf-24",
    S0=initial_state,
    init_args=init_args,
    exog_params=exog_params,
    T=T,
    t0=t0,
    seed=0
)

theta_low = 10
theta_high = 20

BLSH_test_policy = BLSH(
    policy_name="BLSH-Hyperparam-Search",
    model=BLSH_test_model,
    theta_low=theta_low,
    theta_high=theta_high,
    verbose=False
)

In [23]:
theta_low_values = np.linspace(300, 40, 53)  
theta_high_values = np.linspace(60, 300, 49)  

valid_combinations = [(low, high) for low in theta_low_values for high in theta_high_values if low < high]

BLSH_grid = {
    "theta_low": np.array([combo[0] for combo in valid_combinations]),
    "theta_high": np.array([combo[1] for combo in valid_combinations])
}

result = grid_search_PFA(BLSH_grid, BLSH_test_policy, n_iterations=1)
print("Best Parameters:", result["best_parameters"])
print("Best Performance:", result["best_performance"])

Best Parameters: {'theta_low': np.float64(75.0), 'theta_high': np.float64(80.0)}
Best Performance: 7082.379390789476


### Scenarios 

Scenario grid for optimization

- num scenarios: 5, 10
- scenario train length: 0%, 15%, 25%
- noise: 0%, 15%, 25%

In [1]:
import pandas as pd
import numpy as np

num_scenarios_list = [5, 10, 50]
scenario_train_length_list = [0.25, 0.50, 1.0]
noise_list = [0, 0.15, 0.25, 0.35]

policy_train_files = {
    0.0: "./Data/policy_train.csv",
    0.15: "./Data/policy_train_all_feat_noise015.csv",
    0.25: "./Data/policy_train_all_feat_noise025.csv",
    0.35: "./Data/policy_train_all_feat_noise035.csv"
}

policy_test_files = {
    0.0: "./Data/policy_test.csv",
    0.15: "./Data/policy_test_all_feat_noise015.csv",
    0.25: "./Data/policy_test_all_feat_noise025.csv",
    0.35: "./Data/policy_test_all_feat_noise035.csv"
}

def load_and_reshape(file_path, keep_first_column=False):
    df = pd.read_csv(file_path)
    
    if not keep_first_column:
        df.drop(columns=["0"], inplace=True)
    
    array = df.to_numpy()
    num_cols = array.shape[1]
    new_length = (array.shape[0] // 24) * 24
    array = array[:new_length, :]
    reshaped_array = array.reshape(new_length // 24, 24, num_cols).transpose(2, 0, 1)
    
    return reshaped_array


In [41]:
from joblib import Parallel, delayed
import numpy as np
import pandas as pd
from Models.EnergyStorageModel import EnergyStorageModel as ESM
from Models.Policies.PFA import BuyLowSellHigh as BLSH
from Models.BaseClasses.Util import grid_search_PFA 

def run_scenario(num_scenarios, train_length, noise):
    reshaped_policy_train = load_and_reshape(policy_train_files[noise], keep_first_column=False)
    reshaped_policy_test = load_and_reshape(policy_test_files[0.0], keep_first_column=True)
    scenario_test_data = reshaped_policy_test[0]  # Feste Test-Daten

    train_length_idx = int(train_length * reshaped_policy_train.shape[1])
    chosen_train_scenarios = np.random.choice(reshaped_policy_train.shape[0], num_scenarios, replace=False)
    reshaped_policy_train_selected = reshaped_policy_train[chosen_train_scenarios, :, :]

    init_args = {"eta": 0.95, "Rmax": 600, "max_load_per_hour": 50}
    t0 = 0

    # **1. Test-Modell einmal erstellen**
    test_model = ESM(
        model_name="cnf-24",
        S0={"energy_amount": 300, "price": scenario_test_data[0]},
        init_args=init_args,
        exog_params={"hist_price": scenario_test_data[1:]},
        T=len(scenario_test_data[1:]),
        t0=t0,
        seed=0
    )

    # **2. Grid-Search für Test-Policy einmal ausführen**
    BLSH_test_policy = BLSH(
        policy_name="BLSH-On-Policy-Hyperparam-Search",
        model=test_model,
        theta_low=10,
        theta_high=20,
        verbose=False
    )

    theta_low_values = np.linspace(300, 40, 53)
    theta_high_values = np.linspace(60, 300, 49)
    valid_combinations = [(low, high) for low in theta_low_values for high in theta_high_values if low < high]

    BLSH_grid = {
        "theta_low": np.array([combo[0] for combo in valid_combinations]),
        "theta_high": np.array([combo[1] for combo in valid_combinations])
    }

    # **3. Grid-Search einmal für Test-Policy**
    result_test = grid_search_PFA(BLSH_grid, BLSH_test_policy, n_iterations=1)
    best_test_performance = result_test["best_performance"]

    scenario_results = []

    # **4. Szenarien durchgehen (nur fürs Training)**
    for scenario_idx in range(num_scenarios):
        scenario_train_data = reshaped_policy_train_selected[scenario_idx, :, :]

        train_model = ESM(
            model_name="cnf-24",
            S0={"energy_amount": 300, "price": scenario_train_data[0]},
            init_args=init_args,
            exog_params={"hist_price": scenario_train_data[1:train_length_idx]},
            T=len(scenario_train_data[1:train_length_idx]),
            t0=t0,
            seed=0
        )

        BLSH_train_policy = BLSH(
            policy_name="BLSH-Off-policy-Hyperparam-Search",
            model=train_model,
            theta_low=10,
            theta_high=20,
            verbose=False
        )

        # **5. Grid-Search für Training-Policy**
        result_train = grid_search_PFA(BLSH_grid, BLSH_train_policy, n_iterations=1)

        best_train_performance = result_train["best_performance"]
        best_train_theta_low = result_train["best_parameters"]["theta_low"]
        best_train_theta_high = result_train["best_parameters"]["theta_high"]

        best_test_performance_for_train_params = max(result_test["all_runs"][
            (result_test["all_runs"]["theta_low"] == best_train_theta_low) &
            (result_test["all_runs"]["theta_high"] == best_train_theta_high)
        ]["performance"])

        scenario_results.append({
            "theta_low_train": best_train_theta_low,
            "theta_high_train": best_train_theta_high,
            "train_performance": best_train_performance,
            "test_performance": best_test_performance,
            "test_performance_best_train_params": best_test_performance_for_train_params
        })

    # **6. Berechnung der Durchschnittswerte**
    avg_theta_low_train = np.average(
        [res["theta_low_train"] for res in scenario_results],
        weights=[res["train_performance"] for res in scenario_results]
    )
    avg_theta_high_train = np.average(
        [res["theta_high_train"] for res in scenario_results],
        weights=[res["train_performance"] for res in scenario_results]
    )

    highest_contribution_idx = np.argmax([res["train_performance"] for res in scenario_results])
    best_contribution_theta_low = scenario_results[highest_contribution_idx]["theta_low_train"]
    best_contribution_theta_high = scenario_results[highest_contribution_idx]["theta_high_train"]

    # **7. Evaluierung mit besten Trainingsparametern**
    BLSH_eval_best_contrib = BLSH(
        policy_name="Eval best contribution params on test",
        model=test_model,
        theta_low=best_contribution_theta_low,
        theta_high=best_contribution_theta_high,
        verbose=False
    )

    test_performance_best_contrib_params = BLSH_eval_best_contrib.run_policy()

    # **8. Evaluierung mit Durchschnittswerten**
    BLSH_eval_avg = BLSH(
        policy_name="Eval avg train params on test",
        model=test_model,
        theta_low=avg_theta_low_train,
        theta_high=avg_theta_high_train,
        verbose=False
    )

    test_performance_avg_params = BLSH_eval_avg.run_policy()

    norm_performance_test_train_vs_test_best = test_performance_best_contrib_params / best_test_performance if best_test_performance != 0 else np.nan

    return {
        "num_scenarios": num_scenarios,
        "train_length": train_length,
        "noise": noise,
        "theta_low_train": best_contribution_theta_low,
        "theta_high_train": best_contribution_theta_high,
        "train_performance": scenario_results[highest_contribution_idx]["train_performance"],
        "test_performance_train_params": test_performance_best_contrib_params,
        "theta_low_test": result_test["best_parameters"]["theta_low"],
        "theta_high_test": result_test["best_parameters"]["theta_high"],
        "test_performance_test_params": best_test_performance,
        "theta_high_train_avg": avg_theta_high_train,
        "theta_low_train_avg": avg_theta_low_train,
        "test_performance_test_params_avg": test_performance_avg_params,
        "norm_performance_test_train_vs_test_best": norm_performance_test_train_vs_test_best
    }


# **Joblib Parallelisierung**
results = Parallel(n_jobs=-1)(delayed(run_scenario)(num_scenarios, train_length, noise)
                              for num_scenarios in num_scenarios_list
                              for train_length in scenario_train_length_list
                              for noise in noise_list)

# In ein DataFrame umwandeln
results_df = pd.DataFrame(results)

global_best_test_performance = results_df["test_performance_test_params"].max() if not results_df.empty else np.nan
results_df["norm_performance_global_best_test"] = results_df["test_performance_train_params"] / global_best_test_performance
results_df.loc[results_df["test_performance_train_params"] == 0, "norm_performance_global_best_test"] = np.nan

In [70]:
results_df.sort_values("norm_performance_global_best_test", ascending=False, inplace=True)
results_df

,num_scenarios,train_length,noise,theta_low_train,theta_high_train,train_performance,test_performance_train_params,theta_low_test,theta_high_test,test_performance_test_params,theta_high_train_avg,theta_low_train_avg,test_performance_test_params_avg,norm_performance_test_train_vs_test_best,norm_performance_global_best_test
11,5,1.00,0.35,95.0,100.0,5681.414636,5897.079722,75.0,80.0,7082.379391,106.873442,101.873442,5066.898425,0.832641,0.832641
10,5,1.00,0.25,100.0,105.0,4668.726493,5326.501824,75.0,80.0,7082.379391,114.894340,109.894340,4215.895417,0.752078,0.752078
35,50,1.00,0.35,105.0,110.0,5803.770672,4662.482818,75.0,80.0,7082.379391,112.498038,107.498038,4416.997655,0.658322,0.658322
34,50,1.00,0.25,105.0,110.0,4920.564480,4662.482818,75.0,80.0,7082.379391,120.585992,115.585992,3734.347917,0.658322,0.658322
23,10,1.00,0.35,105.0,110.0,5708.951473,4662.482818,75.0,80.0,7082.379391,113.502294,108.502294,4342.302053,0.658322,0.658322
22,10,1.00,0.25,115.0,120.0,4723.354673,3807.757438,75.0,80.0,7082.379391,112.518130,107.518130,4419.992055,0.537638,0.537638
4,5,0.50,0.00,285.0,290.0,4605.892825,316.357655,75.0,80.0,7082.379391,297.923800,292.923800,305.170955,0.044668,0.044668
25,50,0.25,0.15,295.0,300.0,2823.592846,302.948430,75.0,80.0,7082.379391,299.505681,294.123480,305.170955,0.042775,0.042775
32,50,1.00,0.00,295.0,300.0,4978.755974,302.948430,75.0,80.0,7082.379391,299.897709,294.897709,305.170955,0.042775,0.042775
26,50,0.25,0.25,295.0,300.0,2854.333927,302.948430,75.0,80.0,7082.379391,299.599764,294.094432,305.170955,0.042775,0.042775


In [72]:
results_df.to_csv("Results_off_policy_blsh_grid_search_real_test.csv", index=False)

In [44]:
final_model = ESM(
    model_name="cnf-24",
    S0={"energy_amount": 300, "price": reshaped_policy_test_0[0][0]},
    init_args={"eta": 0.95, "Rmax": 600, "max_load_per_hour": 50},
    exog_params={"hist_price": reshaped_policy_test_0[0][1:]},
    T=len(reshaped_policy_test_0[0][1:]),
    t0=0,
    seed=0
)   

final_policy = BLSH(
    policy_name="Final BLSH",
    model=final_model,
    theta_low=results_df.iloc[0]["theta_low_train"],
    theta_high=results_df.iloc[0]["theta_high_train"],
    verbose=False
)

final_policy.run_policy()

5897.079722368419

In [2]:
from joblib import Parallel, delayed
import numpy as np, pandas as pd
from Models.EnergyStorageModel import EnergyStorageModel as ESM
from Models.Policies.PFA import BuyLowSellHigh as BLSH
from Models.BaseClasses.Util import grid_search_PFA

NUM_SCENARIOS = 50  # Anzahl der Szenarien pro Noise-Level

def run_scenario(noise):
    reshaped_policy_train = load_and_reshape(policy_train_files[noise], keep_first_column=False)
    reshaped_policy_test = load_and_reshape(policy_test_files[0.0], keep_first_column=True)
    scenario_test_data = reshaped_policy_test[0]  # Feste Test-Daten für alle Szenarien

    init_args = {"eta": 0.95, "Rmax": 600, "max_load_per_hour": 50}
    t0 = 0

    # **Test-Modell einmal erstellen**
    test_model = ESM(
        model_name="cnf-24",
        S0={"energy_amount": 300, "price": scenario_test_data[0]},
        init_args=init_args,
        exog_params={"hist_price": scenario_test_data[1:]},
        T=len(scenario_test_data[1:]),
        t0=t0,
        seed=0
    )

    BLSH_test_policy = BLSH(
        policy_name="BLSH-On-Policy-Hyperparam-Search",
        model=test_model,
        theta_low=10,
        theta_high=20,
        verbose=False
    )

    theta_low_values = np.linspace(300, 40, 53)
    theta_high_values = np.linspace(60, 300, 49)
    valid_combinations = [(low, high) for low in theta_low_values for high in theta_high_values if low < high]

    BLSH_grid = {
        "theta_low": np.array([combo[0] for combo in valid_combinations]),
        "theta_high": np.array([combo[1] for combo in valid_combinations])
    }

    # **Grid-Search für Test-Policy**
    result_test = grid_search_PFA(BLSH_grid, BLSH_test_policy, n_iterations=1)

    all_results = []
    all_theta_low = []
    all_theta_high = []
    all_train_performance = []
    all_test_performance = []

    # **50 Trainings-Szenarien durchlaufen**
    chosen_train_scenarios = np.random.choice(reshaped_policy_train.shape[0], NUM_SCENARIOS, replace=False)

    for scenario_idx in chosen_train_scenarios:
        scenario_train_data = reshaped_policy_train[scenario_idx, :, :]

        train_model = ESM(
            model_name="cnf-24",
            S0={"energy_amount": 300, "price": scenario_train_data[0]},
            init_args=init_args,
            exog_params={"hist_price": scenario_train_data[1:]},
            T=len(scenario_train_data[1:]),
            t0=t0,
            seed=0
        )

        BLSH_train_policy = BLSH(
            policy_name="BLSH-Off-policy-Hyperparam-Search",
            model=train_model,
            theta_low=10,
            theta_high=20,
            verbose=False
        )

        # **Grid-Search für Trainings-Policy**
        result_train = grid_search_PFA(BLSH_grid, BLSH_train_policy, n_iterations=1)

        for theta_low, theta_high in valid_combinations:
            # Train Performance holen
            train_performance = result_train["all_runs"][
                (result_train["all_runs"]["theta_low"] == theta_low) & 
                (result_train["all_runs"]["theta_high"] == theta_high)
            ]["performance"].values[0]

            # Test Performance holen
            test_performance = result_test["all_runs"][
                (result_test["all_runs"]["theta_low"] == theta_low) & 
                (result_test["all_runs"]["theta_high"] == theta_high)
            ]["performance"].values[0]

            # Ergebnisse speichern
            all_results.append({
                "noise": noise,
                "scenario_idx": scenario_idx,
                "theta_low": theta_low,
                "theta_high": theta_high,
                "train_performance": train_performance,
                "test_performance": test_performance
            })

            all_theta_low.append(theta_low)
            all_theta_high.append(theta_high)
            all_train_performance.append(train_performance)
            all_test_performance.append(test_performance)

    # **Keine Mittelung der Hyperparameter mehr, wir haben jetzt alle Performance-Werte**
    return all_results

# **Parallelisierung für alle Noise-Werte**
results = Parallel(n_jobs=-1)(delayed(run_scenario)(noise) for noise in noise_list)

# **Flatten der Listen in ein DataFrame**
results_flat = [entry for sublist in results for entry in sublist]
results_df = pd.DataFrame(results_flat)

# **Normierte Performance berechnen**
global_best_test_performance = results_df["test_performance"].max()
results_df["norm_performance"] = results_df["test_performance"] / global_best_test_performance
results_df.loc[results_df["test_performance"] == 0, "norm_performance"] = np.nan

In [3]:
results_df.to_csv("Results_off_policy_blsh_grid_search_real_test_all_noises.csv", index=False)

In [26]:
results_df

,noise,scenario_idx,theta_low,theta_high,train_performance,test_performance,norm_performance
0,0.00,14,295.0,300.0,4760.865814,302.948430,0.042775
1,0.00,14,290.0,295.0,4701.393359,312.797505,0.044166
2,0.00,14,290.0,300.0,4698.472028,302.948430,0.042775
3,0.00,14,285.0,290.0,4614.734625,316.357655,0.044668
4,0.00,14,285.0,295.0,4654.489333,312.797505,0.044166
...,...,...,...,...,...,...,...
274395,0.35,91,40.0,280.0,1032.157779,570.748007,0.080587
274396,0.35,91,40.0,285.0,987.566767,559.247557,0.078963
274397,0.35,91,40.0,290.0,999.277121,521.496607,0.073633
274398,0.35,91,40.0,295.0,983.398042,500.594953,0.070682


In [44]:
group_noise = results_df.groupby(["noise", "scenario_idx", "theta_low", "theta_high"]).agg({"train_performance": "mean", "test_performance": "mean", "norm_performance": "mean"}).reset_index()

In [49]:
group_noise[(group_noise["theta_high"] == 110) & (group_noise["theta_low"] == 105)]

,noise,scenario_idx,theta_low,theta_high,train_performance,test_performance,norm_performance
592,0.00,0,105.0,110.0,4113.736696,4662.482818,0.658322
1964,0.00,1,105.0,110.0,4107.879437,4662.482818,0.658322
3336,0.00,5,105.0,110.0,3983.041949,4662.482818,0.658322
4708,0.00,6,105.0,110.0,4039.452740,4662.482818,0.658322
6080,0.00,7,105.0,110.0,4025.214910,4662.482818,0.658322
...,...,...,...,...,...,...,...
268132,0.35,92,105.0,110.0,5303.856951,4662.482818,0.658322
269504,0.35,93,105.0,110.0,5267.512102,4662.482818,0.658322
270876,0.35,95,105.0,110.0,5518.484778,4662.482818,0.658322
272248,0.35,97,105.0,110.0,5566.996861,4662.482818,0.658322


In [30]:
group_noise[group_noise["noise"]==0.35].groupby(["theta_low", "theta_high"]).agg({"train_performance": "mean"}).reset_index().sort_values("train_performance", ascending=False)

,theta_low,theta_high,train_performance
592,105.0,110.0,5442.931509
631,110.0,115.0,5442.260192
552,100.0,105.0,5405.631523
669,115.0,120.0,5405.334799
593,105.0,115.0,5377.651263
...,...,...,...
44,40.0,280.0,1158.627957
45,40.0,285.0,1148.720366
46,40.0,290.0,1146.067971
48,40.0,300.0,1140.005654


In [31]:
group_noise[group_noise["noise"]==0.0].groupby(["theta_low", "theta_high"]).agg({"train_performance": "mean"}).reset_index().sort_values("train_performance", ascending=False)

,theta_low,theta_high,train_performance
1371,295.0,300.0,4754.052992
1370,290.0,300.0,4670.135465
1369,290.0,295.0,4653.618131
1368,285.0,300.0,4589.161987
1367,285.0,295.0,4577.284060
...,...,...,...
46,40.0,290.0,1192.039035
42,40.0,270.0,1186.126874
45,40.0,285.0,1184.890012
43,40.0,275.0,1183.939263


In [32]:
group_noise[group_noise["noise"]==0.15].groupby(["theta_low", "theta_high"]).agg({"train_performance": "mean"}).reset_index().sort_values("train_performance", ascending=False)

,theta_low,theta_high,train_performance
1371,295.0,300.0,4059.120751
1370,290.0,300.0,3997.841811
1369,290.0,295.0,3980.413706
1368,285.0,300.0,3930.844700
1367,285.0,295.0,3918.171133
...,...,...,...
41,40.0,265.0,912.681231
45,40.0,285.0,909.688602
44,40.0,280.0,906.945794
42,40.0,270.0,906.597862


In [33]:
group_noise[group_noise["noise"]==0.25].groupby(["theta_low", "theta_high"]).agg({"train_performance": "mean"}).reset_index().sort_values("train_performance", ascending=False)

,theta_low,theta_high,train_performance
592,105.0,110.0,4512.810149
631,110.0,115.0,4508.569329
669,115.0,120.0,4480.988081
552,100.0,105.0,4480.363909
553,100.0,110.0,4449.037886
...,...,...,...
43,40.0,275.0,1018.135679
47,40.0,295.0,1017.727383
44,40.0,280.0,1015.220601
46,40.0,290.0,1012.155315


In [ ]:
best_train_performance_per_noise = results_df.loc[results_df.groupby('noise')['train_performance'].idxmax()]
sorted_best_train_performance = best_train_performance_per_noise.sort_values(by='test_performance', ascending=False)
sorted_best_train_performance

,noise,scenario_idx,theta_low,theta_high,train_performance,test_performance,norm_performance
173613,0.25,75,105.0,110.0,4920.564480,4662.482818,0.658322
251817,0.35,87,105.0,110.0,5803.770672,4662.482818,0.658322
64484,0.00,43,295.0,300.0,4978.755974,302.948430,0.042775
108388,0.15,84,295.0,300.0,4318.908571,302.948430,0.042775
